# KG → LLM Fine-Tuning Evaluation Pipeline

This notebook runs the full evaluation pipeline on **Google Colab with a T4 GPU**:

| Method | What it does | GPU needed? |
|--------|-------------|-------------|
| **Method 1** — SFT Data Quality | Structural audit, SFT pair generation, quality scoring | No (CPU only) |
| **Method 2** — Ablation Study | Fine-tune Model B (KG-structured) & Model C (raw-text), benchmark vs base Model A | **Yes (T4 GPU)** |

### Prerequisites
- A knowledge graph already uploaded to Neo4j (run `make upload` on your local machine first)
- Neo4j credentials (AuraDB free tier works great)
- DeepSeek API key for SFT pair generation

## Step 1: Check GPU

Make sure a GPU is available for Method 2 (fine-tuning). Method 1 runs fine on CPU.

In [ ]:
import torch
import subprocess
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU available: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️  No GPU detected — Method 2 (fine-tuning) will NOT work.")
    print("   Go to Runtime → Change runtime type → select T4 GPU.")

## Step 2: Clone the Repository & Install Dependencies

In [ ]:
# ═══ Clone the repo ═══
# Replace with your actual repo URL
!git clone https://github.com/your-org/hackathon.git
%cd hackathon

# ═══ Install dependencies ═══
# eval-model: transformers, peft, accelerate, bitsandbytes, datasets (for fine-tuning)
# neo4j: Neo4j Python driver (for downloading the KG)
!pip install -e ".[eval-model,neo4j]"

print("\n✅ Dependencies installed. You may need to restart the runtime if prompted.")
print("   Runtime → Restart runtime, then continue from Step 3.")

## Step 3: Set Credentials

Set your **Neo4j connection details** (to download the KG) and **DeepSeek API key** (for SFT pair generation).

> ⚠️ **Never commit secrets to git.** These are set as runtime environment variables only.

In [ ]:
import os

# ═══ Neo4j — download your KG from the cloud ═══
# Use Neo4j AuraDB free tier (https://neo4j.com/cloud/aura/) or your own instance
os.environ["NEO4J_URI"] = "bolt://your-instance.databases.neo4j.io:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "your-neo4j-password"

# ═══ DeepSeek — for LLM-powered SFT pair generation ═══
# Get your key at: https://platform.deepseek.com/api_keys
os.environ["DEEPSEEK_API_KEY"] = "sk-your-deepseek-key"

# ═══ Optional: write a local .env so kg-gen picks it up ═══
with open(".env", "w") as f:
    f.write(f'NEO4J_URI={os.environ["NEO4J_URI"]}\n')
    f.write(f'NEO4J_USER={os.environ["NEO4J_USER"]}\n')
    f.write(f'NEO4J_PASSWORD={os.environ["NEO4J_PASSWORD"]}\n')
    f.write(f'DEEPSEEK_API_KEY={os.environ["DEEPSEEK_API_KEY"]}\n')

print("✅ Credentials set and .env written.")
print(f"   Neo4j URI: {os.environ['NEO4J_URI']}")
print(f"   DeepSeek key: {'***' + os.environ['DEEPSEEK_API_KEY'][-4:] if len(os.environ['DEEPSEEK_API_KEY']) > 4 else 'NOT SET'}")

## Step 4: Download the Knowledge Graph from Neo4j

Pulls the full graph (nodes + relationships) from Neo4j and saves it as `knowledge_graph.json` — the format expected by the evaluation pipeline.

In [ ]:
import subprocess, os, json
from pathlib import Path

# Ensure output directory exists
output_dir = Path("generated_KGs/output")
output_dir.mkdir(parents=True, exist_ok=True)
kg_path = output_dir / "knowledge_graph.json"

# Download from Neo4j
print("Downloading knowledge graph from Neo4j...")
result = subprocess.run(
    ["kg-gen", "neo4j-download", "-o", str(kg_path)],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ Download failed:")
    print(result.stderr)
    print("\nTroubleshooting:")
    print("  - Is your Neo4j instance running and accessible?")
    print("  - Did you set NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD correctly?")
    print("  - Did you upload the KG first? (make upload on your local machine)")
    raise SystemExit(1)

print(result.stdout)

# Quick sanity check
if kg_path.exists():
    with open(kg_path) as f:
        data = json.load(f)
    stats = data.get("stats", {})
    print(f"\n✅ KG downloaded successfully!")
    print(f"   Nodes: {stats.get('num_nodes', '?'):,}")
    print(f"   Edges: {stats.get('num_edges', '?'):,}")
    print(f"   Triples: {stats.get('num_triples', '?'):,}")
    print(f"   File size: {kg_path.stat().st_size / 1e6:.1f} MB")
else:
    print("❌ knowledge_graph.json not found after download.")
    raise FileNotFoundError(str(kg_path))

## Step 5: Method 1 — SFT Data Quality Assessment

Runs on CPU. Checks graph health, generates SFT training pairs, and evaluates their quality.

**⏱ ~30 seconds**

In [ ]:
import subprocess, json
from pathlib import Path

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 1: SFT Data Quality Assessment")
print("=" * 60)

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "1", "--kg", kg_path],
    capture_output=True, text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:2000])

# Show key results from the output files
method1_dir = Path("output_eval/method1")

# Structural audit summary
audit_path = method1_dir / "structural_audit.json"
if audit_path.exists():
    with open(audit_path) as f:
        audit = json.load(f)
    print(f"\n📊 Structural Audit — Health Score: {audit.get('overall_health_score', '?')}/100")
    print(f"   Verdict: {audit.get('verdict', '?')}")

# SFT quality summary
quality_path = method1_dir / "sft_quality_report.json"
if quality_path.exists():
    with open(quality_path) as f:
        quality = json.load(f)
    print(f"\n📊 SFT Quality — Overall Score: {quality.get('overall_score', '?'):.3f}")
    print(f"   Verdict: {quality.get('verdict', '?')}")

print(f"\n✅ Method 1 complete — full results in {method1_dir}/")

## Step 6: Method 2 — Train Model B (KG-Structured QA)

Fine-tunes Qwen2.5 on QA pairs generated from the knowledge graph. The KG structure should improve multi-hop reasoning and factual consistency.

**⏱ ~10–15 minutes on T4**

In [ ]:
import subprocess

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 2: Training MODEL B (KG-structured QA)")
print("=" * 60)
print("This fine-tunes Qwen2.5 on multi-hop QA pairs from the KG.")
print("Expected time: ~10-15 min on T4 GPU\n")

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "2", "--kg", kg_path, "model=b"],
    capture_output=False,  # stream to notebook output
    text=True
)

if result.returncode == 0:
    print("\n✅ Model B training complete!")
    print("   Checkpoint saved to output_eval/method2/")
else:
    print(f"\n❌ Model B training failed with exit code {result.returncode}")
    print("   Common issues:")
    print("   - Out of VRAM → switch to a smaller model (see Step 8)")
    print("   - Missing dependencies → re-run Step 2")

## Step 7: Method 2 — Train Model C (Raw-Text QA)

Fine-tunes the same base model on flat QA pairs extracted directly from source documents (no KG structure). This is the **control group** — isolating the effect of KG structure.

**⏱ ~10–15 minutes on T4**

In [ ]:
import subprocess

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 2: Training MODEL C (raw-text QA — control)")
print("=" * 60)
print("This fine-tunes Qwen2.5 on flat QA pairs from source documents.")
print("Expected time: ~10-15 min on T4 GPU\n")

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "2", "--kg", kg_path, "model=c"],
    capture_output=False,
    text=True
)

if result.returncode == 0:
    print("\n✅ Model C training complete!")
    print("   Checkpoint saved to output_eval/method2/")
else:
    print(f"\n❌ Model C training failed with exit code {result.returncode}")

## Step 8: Method 2 — Benchmark (A vs B vs C)

Evaluates all three models on the same test set:

| Model | Training Data | Hypothesis |
|-------|--------------|------------|
| **A** | None (base Qwen2.5) | Baseline |
| **B** | KG-structured QA pairs | KG improves multi-hop reasoning |
| **C** | Raw-text QA pairs | Same facts, no graph structure |

**⏱ ~2–5 minutes on T4**

In [ ]:
import subprocess, json
from pathlib import Path

kg_path = "generated_KGs/output/knowledge_graph.json"

print("=" * 60)
print("METHOD 2: Benchmark — Model A (base) vs B (KG) vs C (raw)")
print("=" * 60)

result = subprocess.run(
    ["python", "evaluation/run_eval.py", "--method", "2", "--kg", kg_path, "model=a"],
    capture_output=False,
    text=True
)

if result.returncode == 0:
    print("\n✅ Benchmark complete!")
else:
    print(f"\n⚠️  Benchmark exited with code {result.returncode}")

# ── Try to find and display benchmark results ──
results_dir = Path("output_eval/method2")
benchmark_files = sorted(results_dir.rglob("*benchmark*.json")) if results_dir.exists() else []
results_files = sorted(results_dir.rglob("*results*.json")) if results_dir.exists() else []

for bf in benchmark_files + results_files:
    print(f"\n📊 Results from {bf}:")
    with open(bf) as f:
        data = json.load(f)
    # Pretty-print key metrics
    if isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, (int, float, str)):
                print(f"   {k}: {v}")
            elif isinstance(v, dict) and len(v) <= 10:
                for k2, v2 in v.items():
                    if isinstance(v2, (int, float)):
                        print(f"   {k}.{k2}: {v2:.4f}" if isinstance(v2, float) else f"   {k}.{k2}: {v2}")

print(f"\n✅ Full benchmark results in {results_dir}/")

## Step 9 (Optional): Override Model Size

If you run out of VRAM on the free T4 (~16 GB), switch to a smaller model. Run this cell **before** Steps 6–7.

| Model | 4-bit VRAM | Speed | Recommended for |
|-------|-----------|-------|-----------------|
| `Qwen/Qwen2.5-0.5B-Instruct` | ~1 GB | ⚡ Fast | Fastest iteration |
| `Qwen/Qwen2.5-1.5B-Instruct` | ~2 GB | ⚡ Fast | **Default** — good balance |
| `Qwen/Qwen2.5-3B-Instruct` | ~4 GB | 🐢 Moderate | Better quality |
| `Qwen/Qwen2.5-7B-Instruct` | ~6 GB | 🐢 Slower | Best quality |

Or use a 4-bit quantized version to save VRAM:
- `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`
- `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`

In [ ]:
# ═══ Choose your model ═══
# Uncomment one of the lines below:

# MODEL = "Qwen/Qwen2.5-0.5B-Instruct"    # smallest, fastest
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"      # default — good balance
# MODEL = "Qwen/Qwen2.5-3B-Instruct"      # better quality
# MODEL = "Qwen/Qwen2.5-7B-Instruct"      # best quality, needs ~6 GB VRAM

# ═══ Or use a 4-bit quantized version ═══
# MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

import os
os.environ["FINETUNE_MODEL"] = MODEL

print(f"✅ Using model: {MODEL}")
print("Re-run Steps 6-8 with this model selection.")

## Step 10: Download Results

After everything completes, zip and download the results folder.

In [ ]:
import shutil
from pathlib import Path

# Zip the results
output_zip = "evaluation_results.zip"
shutil.make_archive("evaluation_results", "zip", "output_eval")

print(f"✅ Results zipped to {output_zip}")
print(f"   Size: {Path(output_zip).stat().st_size / 1e6:.1f} MB")

# Download to your local machine
from google.colab import files
files.download(output_zip)

# Also offer individual file downloads
print("\nYou can also browse individual files:")
for f in sorted(Path("output_eval").rglob("*.json")):
    print(f"   output_eval/{f.relative_to('output_eval')}")

## Troubleshooting

<details>
<summary><b>❌ "No module named 'kg_generator'"</b></summary>

Run `!pip install -e ".[eval-model,neo4j]"` again — the package may not be installed in editable mode.
</details>

<details>
<summary><b>❌ Neo4j connection refused</b></summary>

- Make sure your Neo4j instance is running and accessible from the internet.
- AuraDB free tier: check the instance is active in the Neo4j Aura Console.
- Self-hosted: ensure port 7687 is open. Consider using AuraDB for simplicity.
</details>

<details>
<summary><b>❌ CUDA out of memory during fine-tuning</b></summary>

- Switch to a smaller model (Step 9).
- Reduce batch size by editing `eval_config.yaml`: set `per_device_train_batch_size: 1`
- Use a 4-bit quantized model: `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`
</details>

<details>
<summary><b>❌ DeepSeek API key not working</b></summary>

- Verify your key at https://platform.deepseek.com/api_keys
- The SFT generator falls back to template-based generation without an API key (lower quality but still functional).
</details>